# Salvar e carregar os modelos de dificuldade lexical

Este notebook adapta o exemplo fornecido pela professora para salvar o SVM, modelo com melhor Macro-F1 na avaliação final, e as alternativas Regressão Logística e LDA.

## Ambiente

Execute a partir da pasta `ml/` do projeto, com as dependências instaladas. No Google Colab, envie a pasta `ml/` ou monte o Drive e ajuste os caminhos abaixo.

In [1]:
# Opcional no Google Colab:
# from google.colab import drive
# drive.mount('/content/drive')

import sys
from pathlib import Path
import joblib

# Permite importar alfabetiza_ml sem instalar o pacote (executar a partir da pasta ml/)
sys.path.insert(0, str(Path.cwd() / 'src'))

from alfabetiza_ml.train_and_save_human_models import train_and_save, training_data


## Treinar e salvar

Os três modelos são treinados com 339 palavras humanas consolidadas: 94 fáceis, 141 médias e 104 difíceis.

In [2]:
artifacts = train_and_save()
artifacts


{'manifest': '/Users/torquato/Documents/GitHub/DSM-P5-G01-2026-02/ml/artifacts/modelos_dificuldade_humana_manifest.json',
 'models': {'SVM': '/Users/torquato/Documents/GitHub/DSM-P5-G01-2026-02/ml/artifacts/svm_dificuldade_humana.joblib',
  'LR': '/Users/torquato/Documents/GitHub/DSM-P5-G01-2026-02/ml/artifacts/lr_dificuldade_humana.joblib',
  'LDA': '/Users/torquato/Documents/GitHub/DSM-P5-G01-2026-02/ml/artifacts/lda_dificuldade_humana.joblib'},
 'source_corpus': '/Users/torquato/Documents/GitHub/DSM-P5-G01-2026-02/ml/data/Corpus_SANTOS_TONI_FDC_tratado_sem_repeticoes_final.xlsx',
 'training_rows': 339,
 'class_counts': {'medio': 141, 'dificil': 104, 'facil': 94},
 'feature_names': ['syllable_count',
  'complex_syllable_count',
  'cluster_count',
  'max_shape_length',
  'coda_count',
  'glide_count',
  'word_length',
  'accented',
  'digraph_count',
  'complex_grapheme_count',
  'log_frequency'],
 'random_state': 7}

## Carregar o SVM

Equivale ao `joblib.load(...)` do exemplo da professora. Este foi o melhor modelo na avaliação final.

In [3]:
svm_path = Path(artifacts['models']['SVM'])
loaded_svm = joblib.load(svm_path)
svm_model = loaded_svm['model']

X, y, metadata = training_data()
predicoes_svm = svm_model.predict(X)

print('Modelo salvo:', type(svm_model).__name__, '->', [type(etapa).__name__ for _, etapa in svm_model.steps])
print('Algoritmo final:', svm_model.steps[-1][1])
print('Linhas usadas no treino:', metadata['training_rows'])
print('Primeiras previsões:', predicoes_svm[:10])


Modelo salvo: Pipeline -> ['StandardScaler', 'SVC']
Algoritmo final: SVC(class_weight='balanced', random_state=7)
Linhas usadas no treino: 339
Primeiras previsões: ['facil' 'dificil' 'medio' 'dificil' 'medio' 'dificil' 'dificil' 'medio'
 'facil' 'facil']


## Usar o modelo vencedor em palavras do corpus

Carrega o SVM salvo e classifica palavras do corpus tratado, sem treinar novamente.

In [4]:
import pandas as pd
from alfabetiza_ml.benchmark_human_ratings import CORPUS, features

corpus = pd.read_excel(CORPUS).drop_duplicates('PALAVRA')
exemplos = corpus[corpus['PALAVRA'].isin(['casa', 'bola', 'prato', 'abacaxi', 'vermelho', 'brincar'])].copy()
exemplos['PREVISAO_SVM'] = svm_model.predict(features(exemplos))
exemplos[['PALAVRA', 'ESTSILABICA', 'PREVISAO_SVM']]


,PALAVRA,ESTSILABICA,PREVISAO_SVM
2,abacaxi,VCCV,dificil
971,bola,&CV-CV%,facil
1076,brincar,&CCV-CVC%,medio
1353,casa,&CV-CV%,facil
5816,prato,&CCV-CV%,medio
7450,vermelho,&CVC-CV-CV%,dificil


## Rodar no dataset principal (7.649 palavras)

Aplica o SVM salvo a todo o corpus tratado final e mostra quantas palavras foram classificadas em cada dificuldade.

In [5]:
corpus_completo = pd.read_excel(CORPUS).drop_duplicates('PALAVRA').copy()
corpus_completo['PREVISAO_SVM'] = svm_model.predict(features(corpus_completo))

print('Palavras classificadas:', len(corpus_completo))
print(corpus_completo['PREVISAO_SVM'].value_counts().to_string())
corpus_completo[['PALAVRA', 'ESTSILABICA', 'FREQ', 'PREVISAO_SVM']].head(15)


Palavras classificadas: 7649
PREVISAO_SVM
medio      3648
dificil    3509
facil       492


,PALAVRA,ESTSILABICA,FREQ,PREVISAO_SVM
0,a,&V%,10546,facil
1,abacate,&V-CV-CV-CV%,3,medio
2,abacaxi,VCCV,9,dificil
3,abaixa,&V-CVG-CV%,10,dificil
4,abaixada,&V-CVG-CV-CV%,1,dificil
5,abaixando,&V-CVG-CV-CV%,1,dificil
6,abaixar,&V-CVG-CVC%,9,dificil
7,abaixei,&V-CVG-CVG%,1,dificil
8,abaixo,&V-CVG-CV%,2,dificil
9,abandonada,&V-CV-CV-CV-CV%,3,dificil


## Teste dos seis modelos (80% treino / 20% teste)

Repete a comparação oficial com as 339 palavras rotuladas por humanos. O teste usa 68 palavras que os modelos não viram no treino.

In [6]:
from alfabetiza_ml.benchmark_human_ratings import run

relatorio = run()
tabela = pd.DataFrame(relatorio['results'])[['model', 'accuracy', 'macro_f1']]
tabela[['accuracy', 'macro_f1']] = (tabela[['accuracy', 'macro_f1']] * 100).round(2)
print('Treino:', relatorio['train_rows'], '| Teste:', relatorio['test_rows'])
tabela


Treino: 271 | Teste: 68


,model,accuracy,macro_f1
0,SVM,85.29,85.73
1,LR,85.29,85.65
2,LDA,79.41,79.99
3,KNN,75.00,75.53
4,CART,73.53,74.30
5,NB,52.94,51.21


## Carregar a Regressão Logística e o LDA

A Regressão Logística e o LDA foram salvos como alternativas para comparação e reuso.

In [7]:
for model_name in ('LR', 'LDA'):
    loaded = joblib.load(Path(artifacts['models'][model_name]))
    print(model_name, type(loaded['model'].steps[-1][1]).__name__, loaded['model'].predict(X)[:10])


LR LogisticRegression ['facil' 'dificil' 'dificil' 'dificil' 'dificil' 'dificil' 'dificil'
 'medio' 'facil' 'facil']
LDA LinearDiscriminantAnalysis ['facil' 'dificil' 'dificil' 'dificil' 'medio' 'medio' 'dificil' 'medio'
 'facil' 'facil']


## Arquivos gerados

- `ml/artifacts/svm_dificuldade_humana.joblib` — modelo vencedor salvo.
- `ml/artifacts/lr_dificuldade_humana.joblib` e `lda_dificuldade_humana.joblib` — modelos alternativos salvos.
- `ml/artifacts/modelos_dificuldade_humana_manifest.json` — metadados, atributos e origem da base de treinamento.